# 645. Set Mismatch

**Difficulty**: Easy  
**Topics**: Array, Hash Table, Math, Bit Manipulation  
**Link**: [LeetCode 645](https://leetcode.com/problems/set-mismatch/)

---

## Problem Statement

You have a set of integers `s`, which originally contains all the numbers from `1` to `n`. Unfortunately, due to some error, one of the numbers in `s` got duplicated to another number in the set, which results in **repetition of one number** and **loss of another number**.

You are given an integer array `nums` representing the data status of this set after the error.

Find the number that occurs twice and the number that is missing and return them in the form of an array.

### Examples

**Example 1:**
```
Input: nums = [1,2,2,4]
Output: [2,3]
Explanation: 2 is duplicated, 3 is missing.
```

**Example 2:**
```
Input: nums = [1,1]
Output: [1,2]
Explanation: 1 is duplicated, 2 is missing.
```

### Constraints

- `2 <= nums.length <= 10^4`
- `1 <= nums[i] <= 10^4`

---

## Approach 1: Hash Set

### Intuition

1. Use a set to find the duplicate (number seen twice)
2. Compare sum of set vs expected sum to find missing number

### Complexity
- **Time**: O(n)
- **Space**: O(n) - for the set

In [ ]:
def find_error_nums_set(nums: list[int]) -> list[int]:
    """Hash Set approach: O(n) time, O(n) space"""
    n = len(nums)
    seen = set()
    duplicate = -1
    
    # Find duplicate
    for num in nums:
        if num in seen:
            duplicate = num
        seen.add(num)
    
    # Expected sum: 1 + 2 + ... + n = n*(n+1)/2
    expected_sum = n * (n + 1) // 2
    actual_sum = sum(nums)
    
    # missing = expected - actual + duplicate
    # Because: actual = expected - missing + duplicate
    missing = expected_sum - actual_sum + duplicate
    
    return [duplicate, missing]

# Test
print(find_error_nums_set([1, 2, 2, 4]))  # [2, 3]
print(find_error_nums_set([1, 1]))        # [1, 2]

---

## Approach 2: Counting Array

### Intuition

Count occurrences of each number. The one with count 2 is duplicate, count 0 is missing.

### Complexity
- **Time**: O(n)
- **Space**: O(n)

In [ ]:
def find_error_nums_count(nums: list[int]) -> list[int]:
    """Counting approach"""
    n = len(nums)
    count = [0] * (n + 1)  # index 0 unused
    
    for num in nums:
        count[num] += 1
    
    duplicate = missing = -1
    for i in range(1, n + 1):
        if count[i] == 2:
            duplicate = i
        elif count[i] == 0:
            missing = i
    
    return [duplicate, missing]

# Test
print(find_error_nums_count([1, 2, 2, 4]))  # [2, 3]
print(find_error_nums_count([1, 1]))        # [1, 2]

---

## Approach 3: In-Place Marking (O(1) Space)

### Intuition

Use the array itself as a hash map by marking visited indices with negative values.

For each number `num`, mark `nums[|num| - 1]` as negative:
- If already negative → `num` is the duplicate
- After marking, the index with positive value → that index + 1 is missing

### Complexity
- **Time**: O(n)
- **Space**: O(1) - modifies input array

In [ ]:
def find_error_nums_inplace(nums: list[int]) -> list[int]:
    """In-place marking: O(n) time, O(1) space"""
    duplicate = -1
    
    # Mark visited indices as negative
    for num in nums:
        idx = abs(num) - 1  # Convert to 0-indexed
        
        if nums[idx] < 0:
            # Already visited → this is the duplicate
            duplicate = abs(num)
        else:
            nums[idx] = -nums[idx]
    
    # Find missing: the index that's still positive
    missing = -1
    for i in range(len(nums)):
        if nums[i] > 0:
            missing = i + 1  # Convert back to 1-indexed
            break
    
    return [duplicate, missing]

# Test
print(find_error_nums_inplace([1, 2, 2, 4]))  # [2, 3]
print(find_error_nums_inplace([1, 1]))        # [1, 2]

---

## Approach 4: Math (Sum and Sum of Squares)

### Intuition

Use two equations:
1. `sum(nums) - expected_sum = duplicate - missing`
2. `sum(nums²) - expected_sum² = duplicate² - missing²`

From equation 2: `duplicate² - missing² = (duplicate + missing)(duplicate - missing)`

Solve for both values.

### Complexity
- **Time**: O(n)
- **Space**: O(1)

In [ ]:
def find_error_nums_math(nums: list[int]) -> list[int]:
    """Math approach: O(n) time, O(1) space"""
    n = len(nums)
    
    # Expected sums
    expected_sum = n * (n + 1) // 2
    expected_sq_sum = n * (n + 1) * (2 * n + 1) // 6
    
    # Actual sums
    actual_sum = sum(nums)
    actual_sq_sum = sum(x * x for x in nums)
    
    # diff1 = duplicate - missing
    diff1 = actual_sum - expected_sum
    
    # diff2 = duplicate² - missing² = (dup + miss)(dup - miss)
    diff2 = actual_sq_sum - expected_sq_sum
    
    # sum_val = duplicate + missing = diff2 / diff1
    sum_val = diff2 // diff1
    
    # Solve:
    # duplicate - missing = diff1
    # duplicate + missing = sum_val
    duplicate = (diff1 + sum_val) // 2
    missing = (sum_val - diff1) // 2
    
    return [duplicate, missing]

# Test
print(find_error_nums_math([1, 2, 2, 4]))  # [2, 3]
print(find_error_nums_math([1, 1]))        # [1, 2]

---

## Walkthrough

Input: `[1, 2, 2, 4]` (n = 4)

### Using Hash Set Approach:

| Step | num | seen | duplicate |
|------|-----|------|----------|
| 0 | 1 | {1} | - |
| 1 | 2 | {1,2} | - |
| 2 | 2 | {1,2} | **2** (already in set) |
| 3 | 4 | {1,2,4} | 2 |

```
expected_sum = 4 * 5 / 2 = 10
actual_sum = 1 + 2 + 2 + 4 = 9
missing = 10 - 9 + 2 = 3
```

**Result**: `[2, 3]`

In [ ]:
def find_error_nums_verbose(nums: list[int]) -> list[int]:
    """Verbose version showing the process"""
    n = len(nums)
    seen = set()
    duplicate = -1
    
    print(f"Input: {nums}, n = {n}")
    print(f"Expected numbers: 1 to {n}")
    print()
    
    for i, num in enumerate(nums):
        if num in seen:
            duplicate = num
            print(f"Step {i}: num={num} already in seen → duplicate = {duplicate}")
        else:
            print(f"Step {i}: num={num} added to seen")
        seen.add(num)
    
    expected_sum = n * (n + 1) // 2
    actual_sum = sum(nums)
    missing = expected_sum - actual_sum + duplicate
    
    print(f"\nExpected sum (1+2+...+{n}): {expected_sum}")
    print(f"Actual sum: {actual_sum}")
    print(f"Missing = {expected_sum} - {actual_sum} + {duplicate} = {missing}")
    
    return [duplicate, missing]

find_error_nums_verbose([1, 2, 2, 4])

---

## Common Mistakes

1. **Off-by-one errors** - numbers are 1 to n, not 0 to n-1
2. **Integer overflow** - for sum of squares with large n (use Python's arbitrary precision)
3. **Modifying input** - in-place approach changes the array
4. **Wrong order** - return `[duplicate, missing]`, not `[missing, duplicate]`

---

## Related Problems

- [287. Find the Duplicate Number](https://leetcode.com/problems/find-the-duplicate-number/) - Medium
- [268. Missing Number](https://leetcode.com/problems/missing-number/) - Easy
- [442. Find All Duplicates in an Array](https://leetcode.com/problems/find-all-duplicates-in-an-array/) - Medium
- [448. Find All Numbers Disappeared in an Array](https://leetcode.com/problems/find-all-numbers-disappeared-in-an-array/) - Easy

---

## Key Takeaways

| Pattern | When to Use |
|---------|-------------|
| **Hash Set** | Find duplicates in O(n) time |
| **Sum formula** | n*(n+1)/2 for 1 to n |
| **In-place marking** | Use array indices as hash map |
| **Math equations** | Solve for unknowns with sum and sum of squares |